# Latest Gemini 3.1 Flash-Lite (Preview) — 42 fields (per paper)

This notebook scans `outputs/` and, for each `paper_id` (the numeric prefix in filenames), finds the **latest** CSV for:

- method: `direct_llm`
- provider/model tag: `google_gemini-3-1-flash-lite-preview`
- standard fields tag: `42fields`

Latest-ness is determined by the `YYYY-MM-DD` date embedded in the filename.

The notebook is written to run even if `pandas` is not installed (it will fall back to pure-Python).

In [5]:
import csv
import re
from pathlib import Path

try:
    import pandas as pd  # type: ignore
except Exception:  # pragma: no cover
    pd = None

try:
    from IPython.display import display  # type: ignore
except Exception:  # pragma: no cover
    def display(x):
        print(x)


def find_repo_root(start: Path | None = None) -> Path:
    p = start or Path.cwd()
    for _ in range(6):
        if (p / "outputs").exists() and (p / "src").exists():
            return p
        p = p.parent
    return start or Path.cwd()


repo_root = find_repo_root()
outputs_root = repo_root / "outputs"

# Target strings as they appear in your filenames
method_prefix = "direct_llm"
provider_model_tag = "google_gemini-3-1-flash-lite-preview"
n_fields_tag = "42fields"

needle = f"{method_prefix}_{provider_model_tag}_{n_fields_tag}_"

paper_id_re = re.compile(r"^(?P<paper_id>\d+)_")
date_re = re.compile(r"_(?P<date>\d{4}-\d{2}-\d{2})\.csv$")


def scan_latest_direct_llm_per_paper() -> list[dict]:
    latest_by_paper_id: dict[int, dict] = {}

    if not outputs_root.exists():
        raise FileNotFoundError(f"outputs folder not found: {outputs_root}")

    # Each run is stored under a dated directory: outputs/YYYY-MM-DD/
    for dated_dir in sorted(outputs_root.iterdir()):
        if not dated_dir.is_dir():
            continue
        if not re.match(r"^\d{4}-\d{2}-\d{2}$", dated_dir.name):
            continue

        for csv_path in dated_dir.glob("*.csv"):
            name = csv_path.name
            if needle not in name:
                continue

            m_pid = paper_id_re.match(name)
            m_date = date_re.search(name)
            if not m_pid or not m_date:
                continue

            paper_id = int(m_pid.group("paper_id"))
            date = m_date.group("date")  # YYYY-MM-DD, compares lexicographically

            prev = latest_by_paper_id.get(paper_id)
            if prev is None or date > prev["date"]:
                latest_by_paper_id[paper_id] = {
                    "paper_id": paper_id,
                    "date": date,
                    "csv_path": str(csv_path),
                }

    return [latest_by_paper_id[pid] for pid in sorted(latest_by_paper_id.keys())]


latest_rows = scan_latest_direct_llm_per_paper()
print(f"Matched {len(latest_rows)} paper_id(s)")

if pd is not None and latest_rows:
    df_latest = pd.DataFrame(latest_rows).sort_values("paper_id").reset_index(drop=True)
    display(df_latest.head(10))
else:
    display(latest_rows[:10])


Matched 10 paper_id(s)


,paper_id,date,csv_path
0,1,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
1,2,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
2,3,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
3,4,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
4,5,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
5,6,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
6,7,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
7,8,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
8,9,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...
9,10,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...


In [6]:
def count_csv_rows(path: str) -> int:
    # Counts data rows excluding the header.
    with open(path, "r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.reader(f)
        n = 0
        for i, _ in enumerate(reader):
            n = i
    # If the CSV has at least a header line, data rows = total_lines - 1
    return max(0, n)


if not latest_rows:
    print("No matching CSVs found.")
else:
    latest_rows_with_counts = []
    for r in latest_rows:
        rr = dict(r)
        rr["n_records"] = count_csv_rows(rr["csv_path"])
        latest_rows_with_counts.append(rr)

    # Coverage / date summary
    dates = sorted({r["date"] for r in latest_rows_with_counts})
    print(f"Unique latest run dates: {len(dates)}")
    print(f"Earliest latest date: {dates[0] if dates else 'n/a'}")
    print(f"Latest latest date:   {dates[-1] if dates else 'n/a'}")

    latest_rows_with_counts.sort(key=lambda x: x["n_records"], reverse=True)

    if pd is not None:
        df_counts = pd.DataFrame(latest_rows_with_counts)
        display(df_counts.head(10))
    else:
        display(latest_rows_with_counts[:10])


Unique latest run dates: 1
Earliest latest date: 2026-03-25
Latest latest date:   2026-03-25


,paper_id,date,csv_path,n_records
0,1,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,8
1,10,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,8
2,4,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,6
3,5,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,5
4,9,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,5
5,2,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,4
6,6,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,2
7,7,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,2
8,8,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,2
9,3,2026-03-25,/home/com3dian/Github/meta_analysis_agents/out...,1


In [7]:
paper_id_to_preview = latest_rows[0]["paper_id"] if latest_rows else None
print(f"Previewing paper_id={paper_id_to_preview}")

if paper_id_to_preview is None:
    raise SystemExit(0)

row = next(r for r in latest_rows if r["paper_id"] == paper_id_to_preview)
csv_path = row["csv_path"]
print(f"CSV date: {row['date']} | path: {csv_path}")

if pd is not None:
    df = pd.read_csv(csv_path)
    display(df.head())
    print(f"Shape: {df.shape}")
else:
    with open(csv_path, "r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f)
        first = []
        for i, rec in enumerate(reader):
            first.append(rec)
            if i >= 4:
                break
    display(first)


Previewing paper_id=1
CSV date: 2026-03-25 | path: /home/com3dian/Github/meta_analysis_agents/outputs/2026-03-25/1_direct_llm_google_gemini-3-1-flash-lite-preview_42fields_2026-03-25.csv


,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K input IC1,K input IC2,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit
0,1980,1 growing season,Randomized split-plot design,16 April 1980,16 April 1980,NaN,NaN,55.41,12.05,Pisum sativum,...,50,50,50,kg K2O ha-1,Table 3,456,412,NaN,NaN,g DM m-2
1,1980,1 growing season,Randomized split-plot design,16 April 1980,16 April 1980,NaN,NaN,55.41,12.05,Pisum sativum,...,50,50,50,kg K2O ha-1,Table 3,357,550,NaN,NaN,g DM m-2
2,1981,1 growing season,Randomized split-plot design,13 April 1981,13 April 1981,NaN,NaN,55.41,12.05,Pisum sativum,...,50,50,50,kg K2O ha-1,Table 3,437,355,NaN,NaN,g DM m-2
3,1981,1 growing season,Randomized split-plot design,13 April 1981,13 April 1981,NaN,NaN,55.41,12.05,Pisum sativum,...,50,50,50,kg K2O ha-1,Table 3,456,467,NaN,NaN,g DM m-2
4,1982,1 growing season,Randomized split-plot design,7 April 1982,7 April 1982,NaN,NaN,55.41,12.05,Pisum sativum,...,50,50,50,kg K2O ha-1,Table 3,714,473,NaN,NaN,g DM m-2


Shape: (8, 42)


In [8]:
# TP/FP/FN/TN on value presence (assumes GT row order matches predicted row order).

import zipfile
import xml.etree.ElementTree as ET


def is_present(v) -> bool:
    if v is None:
        return False
    s = str(v).strip()
    if not s:
        return False
    s_low = s.lower()
    return s_low not in {'nan', 'none', 'null', 'n/a', 'na'}


def read_prediction_csv(csv_path: str) -> tuple[list[dict], list[str]]:
    with open(csv_path, 'r', encoding='utf-8', errors='ignore', newline='') as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        rows = [dict(r) for r in reader]
    return rows, fieldnames


def load_ground_truth_by_study_id() -> dict[int, list[dict]]:
    gt_path = repo_root / 'data' / 'wopke_100' / 'annotation' / 'wopke100.xlsx'
    if not gt_path.exists():
        raise FileNotFoundError(f'Ground truth Excel not found: {gt_path}')

    # Parse XLSX (zip of XML) without external Excel libraries.
    with zipfile.ZipFile(gt_path, 'r') as zf:
        wb_xml = ET.fromstring(zf.read('xl/workbook.xml'))
        rels_xml = ET.fromstring(zf.read('xl/_rels/workbook.xml.rels'))

        sheet_name = 'labels'
        rid = None
        for sh in wb_xml.iter():
            if sh.tag.endswith('sheet') and sh.attrib.get('name') == sheet_name:
                rid = sh.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
                break
        if rid is None:
            raise ValueError("Could not find sheet 'labels' in wopke100.xlsx")

        target = None
        for rel in rels_xml.iter():
            if rel.tag.endswith('Relationship') and rel.attrib.get('Id') == rid:
                target = rel.attrib.get('Target')
                break
        if target is None:
            raise ValueError(f'Could not find relationship for rid={rid}')

        sheet_path = 'xl/' + target
        sheet_xml = ET.fromstring(zf.read(sheet_path))

        shared_strings: list[str] = []
        if 'xl/sharedStrings.xml' in zf.namelist():
            ss_root = ET.fromstring(zf.read('xl/sharedStrings.xml'))
            for si in ss_root.findall('.//{*}si'):
                parts = []
                for t in si.findall('.//{*}t'):
                    parts.append(t.text or '')
                shared_strings.append(''.join(parts))

        def col_to_index(letters: str) -> int:
            idx = 0
            for ch in letters:
                idx = idx * 26 + (ord(ch) - 64)
            return idx - 1

        def ref_to_col(ref: str) -> int:
            letters = []
            for ch in ref:
                if ch.isalpha():
                    letters.append(ch)
                else:
                    break
            return col_to_index(''.join(letters))

        def cell_value(cel):
            t = cel.attrib.get('t')
            v_el = cel.find('.//{*}v')
            if t == 's':
                if v_el is None or v_el.text is None:
                    return ''
                return shared_strings[int(v_el.text)]
            if t == 'inlineStr':
                t_el = cel.find('.//{*}t')
                return (t_el.text if t_el is not None and t_el.text is not None else '')
            if v_el is None or v_el.text is None:
                return None
            return v_el.text

        header_map: dict[int, object] = {}
        max_col = -1
        for row_el in sheet_xml.iter():
            if not row_el.tag.endswith('row'):
                continue
            r_idx = int(row_el.attrib.get('r', '0'))
            if r_idx != 1:
                continue
            for c_el in row_el:
                if not c_el.tag.endswith('c'):
                    continue
                ref = c_el.attrib.get('r')
                if not ref:
                    continue
                col = ref_to_col(ref)
                header_map[col] = cell_value(c_el)
                max_col = max(max_col, col)

        header_list = [header_map.get(i) for i in range(max_col + 1)]

        deduped: list[str] = []
        seen: dict[str, int] = {}
        for col_name in header_list:
            col_name = col_name if col_name is not None else ''
            if col_name in seen:
                seen[col_name] += 1
                deduped.append(f'{col_name}.{seen[col_name]}')
            else:
                seen[col_name] = 0
                deduped.append(col_name)

        gt_by_id: dict[int, list[dict]] = {}
        for row_el in sheet_xml.iter():
            if not row_el.tag.endswith('row'):
                continue
            r_idx = int(row_el.attrib.get('r', '0'))
            if r_idx < 2:
                continue

            rec_values = [None] * (max_col + 1)
            any_non_none = False

            for c_el in row_el:
                if not c_el.tag.endswith('c'):
                    continue
                ref = c_el.attrib.get('r')
                if not ref:
                    continue
                col = ref_to_col(ref)
                val = cell_value(c_el)
                rec_values[col] = val
                if val is not None:
                    any_non_none = True

            if not any_non_none:
                continue

            rec = dict(zip(deduped, rec_values))

            study_key = 'Study#' if 'Study#' in rec else next((k for k in rec.keys() if k.startswith('Study#')), None)
            if study_key is None:
                continue

            sid_val = rec.get(study_key)
            if sid_val is None:
                continue

            try:
                sid_int = int(float(sid_val))
            except Exception:
                sid_int = int(str(sid_val).strip().split()[0])

            gt_by_id.setdefault(sid_int, []).append(rec)

        return gt_by_id


# --- Load GT and run comparison for all latest prediction CSVs ---
gt_by_id = load_ground_truth_by_study_id()
pred_by_id = {int(r['paper_id']): r['csv_path'] for r in latest_rows}
paper_ids = sorted(pred_by_id.keys())

if not paper_ids:
    raise SystemExit('No prediction CSVs found; re-run discovery cell.')

first_pred_csv = pred_by_id[paper_ids[0]]
_ext_rows0, pred_cols0 = read_prediction_csv(first_pred_csv)

overall_conf = {'TP': 0, 'FP': 0, 'FN': 0, 'TN': 0}
papers_processed = 0

for pid in paper_ids:
    pred_csv_path = pred_by_id[pid]
    pred_rows, pred_cols = read_prediction_csv(pred_csv_path)
    gt_rows = gt_by_id.get(pid, [])

    if gt_rows:
        gt_cols = set(gt_rows[0].keys())
        shared_cols = [c for c in pred_cols if c in gt_cols]
    else:
        shared_cols = pred_cols

    n = max(len(gt_rows), len(pred_rows))
    for i in range(n):
        gt_row = gt_rows[i] if i < len(gt_rows) else {}
        pred_row = pred_rows[i] if i < len(pred_rows) else {}
        for c in shared_cols:
            gt_p = is_present(gt_row.get(c))
            pred_p = is_present(pred_row.get(c))

            if gt_p and pred_p:
                overall_conf['TP'] += 1
            elif gt_p and (not pred_p):
                overall_conf['FN'] += 1
            elif (not gt_p) and pred_p:
                overall_conf['FP'] += 1
            else:
                overall_conf['TN'] += 1

    papers_processed += 1

TP, FP, FN, TN = overall_conf['TP'], overall_conf['FP'], overall_conf['FN'], overall_conf['TN']
precision = TP / (TP + FP) if (TP + FP) else float('nan')
recall = TP / (TP + FN) if (TP + FN) else float('nan')
specificity = TN / (TN + FP) if (TN + FP) else float('nan')
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else float('nan')

print(f'Confusion (cell-level) across {papers_processed} paper_id(s)')
print(overall_conf)
print(f'precision={precision:.4f} recall={recall:.4f} specificity={specificity:.4f} f1={f1:.4f}')

Confusion (cell-level) across 10 paper_id(s)
{'TP': 1069, 'FP': 364, 'FN': 661, 'TN': 129}
precision=0.7460 recall=0.6179 specificity=0.2617 f1=0.6759
